# Design and Simulation of a Slider-Crank Mechanism
### Python-Based Kinematic and Dynamic Analysis Tool
**Author:** Dhruv Chauhan (24JE0497)  
**Discipline:** Mechanical Engineering / Theory of Machines

---
## 1. Overview
This interactive notebook models the complete kinematics and dynamics of an inline or offset slider-crank mechanism. It calculates:
- Exact closed-form piston displacement $s(\theta)$, velocity $v(\theta)$, and acceleration $a(\theta)$
- Connecting rod angular position $\phi$, angular velocity $\omega_c$, and angular acceleration $\alpha_c$
- Dynamic forces: Gas force $F_g$, Reciprocating inertia $F_I$, Piston effort $F_P$, Cylinder side thrust $F_N$
- Turning Moment Diagram (TMD), Indicated Power, Flywheel sizing ($\Delta E, I_{\text{flywheel}}$)
- Fourier Harmonic order vibration decomposition

In [ ]:
# Step 1: Install dependencies and import scientific libraries
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import Union, Dict, Any

print("NumPy version:", np.__version__)
print("Libraries loaded successfully!")

In [ ]:
# Step 2: Define Mechanism Parameters Data Class
@dataclass
class SliderCrankParameters:
    crank_length: float = 0.05       # r: crank radius (meters)
    con_rod_length: float = 0.20     # l: connecting rod length (meters)
    offset: float = 0.0              # e: stroke line offset (meters)
    rpm: float = 100.0               # Crank speed in RPM (default: 100 RPM, range: 0 to 8000 RPM)
    piston_mass: float = 0.85        # m_p: mass of piston assembly (kg)
    con_rod_mass: float = 1.20       # m_rod: mass of connecting rod (kg)
    bore_diameter: float = 0.080     # D: cylinder bore (meters)
    peak_pressure_bar: float = 45.0  # Peak combustion gas pressure (bar)

    @property
    def r(self) -> float: return self.crank_length
    @property
    def l(self) -> float: return self.con_rod_length
    @property
    def e(self) -> float: return self.offset
    @property
    def n(self) -> float: return self.con_rod_length / self.crank_length
    @property
    def omega(self) -> float: return 2.0 * np.pi * self.rpm / 60.0
    @property
    def stroke(self) -> float:
        hyp_tdc = self.l + self.r
        hyp_bdc = self.l - self.r
        return np.sqrt(hyp_tdc**2 - self.e**2) - np.sqrt(hyp_bdc**2 - self.e**2)
    @property
    def piston_area(self) -> float: return np.pi * (self.bore_diameter / 2.0)**2
    @property
    def equivalent_reciprocating_mass(self) -> float:
        # 2-mass equivalent system (1/3 rod mass at piston pin)
        return self.piston_mass + (1.0 / 3.0) * self.con_rod_mass

params = SliderCrankParameters()
print(f"Initialized: Stroke = {params.stroke*1000:.1f} mm, Ratio n = {params.n:.2f}, Speed = {params.rpm} RPM")

In [ ]:
# Step 3: Exact Kinematic & Dynamic Equations Solver
def solve_slider_crank(params: SliderCrankParameters, num_points: int = 360):
    thetas = np.linspace(0, 2.0 * np.pi, num_points, endpoint=False)
    deg = np.degrees(thetas)
    r, l, e = params.r, params.l, params.e
    omega, n = params.omega, params.n
    
    cos_t = np.cos(thetas)
    sin_t = np.sin(thetas)
    
    # Connecting rod angle phi
    sin_phi = np.clip((r * sin_t - e) / l, -1.0, 1.0)
    phi = np.arcsin(sin_phi)
    cos_phi = np.cos(phi)
    dphi_dtheta = (r * cos_t) / (l * cos_phi)
    
    # Piston position & displacement from TDC
    x_B = r * cos_t + l * cos_phi
    x_tdc = np.sqrt((l + r)**2 - e**2)
    s_B = x_tdc - x_B
    
    # Velocity (exact & series approximation)
    v_exact = omega * (-r * sin_t - r * np.tan(phi) * cos_t)
    v_approx = -r * omega * (sin_t + np.sin(2.0 * thetas) / (2.0 * n))
    
    # Acceleration (exact & series approximation)
    d2x = -r * cos_t - r * ((1.0 / (cos_phi**2)) * dphi_dtheta * cos_t - np.tan(phi) * sin_t)
    a_exact = (omega**2) * d2x
    a_approx = -r * (omega**2) * (cos_t + np.cos(2.0 * thetas) / n)
    
    # Dynamics: Gas force (polytropic model)
    p_net = params.peak_pressure_bar * 1e5 * np.exp(-2.5 * (thetas / np.pi))
    comp_mask = thetas > np.pi
    p_net[comp_mask] = 1.013e5 * (1.0 + 8.0 * ((thetas[comp_mask] - np.pi) / np.pi)**2)
    f_gas = p_net * params.piston_area
    
    # Inertia force & net piston effort
    m_rec = params.equivalent_reciprocating_mass
    f_inertia = m_rec * a_exact
    f_piston = f_gas + f_inertia
    
    # Forces on con-rod, side thrust, crank effort & torque
    f_rod = f_piston / np.where(np.abs(cos_phi) < 1e-6, 1e-6, cos_phi)
    f_side = f_piston * np.tan(phi)
    f_tangential = f_piston * np.sin(thetas + phi) / cos_phi
    torque = f_tangential * r
    
    return {
        'thetas': thetas, 'deg': deg,
        'disp': s_B, 'v_exact': v_exact, 'v_approx': v_approx,
        'a_exact': a_exact, 'a_approx': a_approx,
        'f_gas': f_gas, 'f_inertia': f_inertia, 'f_piston': f_piston,
        'f_side': f_side, 'f_rod': f_rod, 'torque': torque
    }

res = solve_slider_crank(params)
print("Simulation completed successfully for 360 crank degrees!")

In [ ]:
# Step 4: Plot Kinematics (Displacement, Velocity & Acceleration)
fig, axs = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
fig.subplots_adjust(hspace=0.25)

axs[0].plot(res['deg'], res['disp'] * 1000.0, 'b-', label='Exact Closed-Form')
axs[0].set_ylabel('Disp (mm)', fontweight='bold')
axs[0].grid(True, linestyle='--', alpha=0.5)
axs[0].legend(loc='upper right')
axs[0].set_title(f'Piston Kinematics — r={params.r*1000:.0f}mm, l={params.l*1000:.0f}mm, {params.rpm:.0f} RPM', fontweight='bold')

axs[1].plot(res['deg'], res['v_exact'], 'g-', label='Exact Velocity')
axs[1].plot(res['deg'], res['v_approx'], 'r--', label='Series Approx')
axs[1].axhline(0, color='k', linewidth=0.8)
axs[1].set_ylabel('Velocity (m/s)', fontweight='bold')
axs[1].grid(True, linestyle='--', alpha=0.5)
axs[1].legend(loc='upper right')

axs[2].plot(res['deg'], res['a_exact'], 'm-', label='Exact Accel')
axs[2].plot(res['deg'], res['a_approx'], 'r--', label='Series Approx')
axs[2].axhline(0, color='k', linewidth=0.8)
axs[2].set_xlabel('Crank Angle θ (degrees)', fontweight='bold')
axs[2].set_ylabel('Accel (m/s²)', fontweight='bold')
axs[2].set_xlim(0, 360)
axs[2].set_xticks(np.arange(0, 361, 45))
axs[2].grid(True, linestyle='--', alpha=0.5)
axs[2].legend(loc='upper right')
plt.show()

In [ ]:
# Step 5: Plot Dynamic Forces & Turning Moment Diagram (TMD)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

# Forces
ax1.plot(res['deg'], res['f_gas'] / 1000.0, color='#E65100', label='Gas Force Fg (kN)')
ax1.plot(res['deg'], res['f_inertia'] / 1000.0, color='#0288D1', label='Inertia FI (kN)')
ax1.plot(res['deg'], res['f_piston'] / 1000.0, color='#2E7D32', linewidth=2.2, label='Piston Effort FP (kN)')
ax1.set_ylabel('Force (kN)', fontweight='bold')
ax1.grid(True, linestyle='--', alpha=0.5)
ax1.legend(loc='upper right')
ax1.set_title('Dynamic Forces Acting on Piston', fontweight='bold')

# Torque
trapz_func = getattr(np, 'trapezoid', getattr(np, 'trapz', None))
t_mean = float(trapz_func(res['torque'], res['thetas']) / (2.0 * np.pi))
power_kw = (t_mean * params.omega) / 1000.0

ax2.plot(res['deg'], res['torque'], color='#C2185B', linewidth=2.2, label='Crankshaft Torque T(θ)')
ax2.axhline(t_mean, color='#1976D2', linewidth=2, linestyle='--', label=f'Mean Torque = {t_mean:.1f} N·m')
ax2.fill_between(res['deg'], res['torque'], t_mean, where=(res['torque'] >= t_mean), color='#4CAF50', alpha=0.3, label='Excess Energy (+)')
ax2.fill_between(res['deg'], res['torque'], t_mean, where=(res['torque'] < t_mean), color='#F44336', alpha=0.3, label='Deficit Energy (-)')
ax2.set_xlabel('Crank Angle θ (degrees)', fontweight='bold')
ax2.set_ylabel('Torque (N·m)', fontweight='bold')
ax2.set_xlim(0, 360)
ax2.set_xticks(np.arange(0, 361, 45))
ax2.grid(True, linestyle='--', alpha=0.5)
ax2.legend(loc='upper right')
ax2.set_title(f'Turning Moment Diagram — Indicated Power: {power_kw:.2f} kW', fontweight='bold')
plt.show()

In [ ]:
# Step 6: Flywheel Sizing & Harmonic Fourier Spectrum
delta_t = res['torque'] - t_mean
d_theta = res['thetas'][1] - res['thetas'][0]
energy_integral = np.cumsum(delta_t) * d_theta
delta_e = float(np.max(energy_integral) - np.min(energy_integral))
c_s = 0.02  # 2% speed fluctuation coefficient
i_flywheel = delta_e / (c_s * (params.omega**2))
r_rim = params.r * 2.5
m_flywheel = i_flywheel / (r_rim**2)

print("=" * 55)
print("  FLYWHEEL SIZING SPECIFICATIONS")
print("=" * 55)
print(f"Mean Engine Torque:           {t_mean:8.1f} N·m")
print(f"Indicated Power Output:       {power_kw:8.2f} kW")
print(f"Fluctuation of Energy (ΔE):   {delta_e:8.1f} Joules")
print(f"Required Flywheel Inertia:    {i_flywheel:8.4f} kg·m² (for Cs=0.02)")
print(f"Flywheel Rim Mass (R={r_rim*1000:.0f}mm):   {m_flywheel:8.2f} kg")
print("=" * 55)